# Routing

JuPedSim assigns each agent a **journey** at spawn time, but routes can be
changed while the simulation is running.  `simulation.switch_agent_journey(agent_id,
new_journey_id, new_stage_id)` redirects an agent to a different journey and
places it at any stage within that journey – useful for dynamic load balancing,
responding to incidents, or splitting crowds at runtime.

For offline path queries (e.g. "what is the shortest path between two points?")
you can use `jps.RoutingEngine` directly, but the most common pattern is the
runtime switch shown below.


## Example – Re-routing agents mid-simulation

Twenty agents start heading toward the top exit.  At iteration 200 the second
half of the agent list is switched to a bottom exit, balancing the egress load.


In [ ]:
import pathlib

import jupedsim as jps
import shapely

trajectory_file = pathlib.Path("routing.sqlite")

geometry = shapely.Polygon([(0, 0), (20, 0), (20, 10), (0, 10)])
simulation = jps.Simulation(
    model=jps.CollisionFreeSpeedModel(),
    geometry=geometry,
    trajectory_writer=jps.SqliteTrajectoryWriter(output_file=trajectory_file),
)

exit_top    = simulation.add_exit_stage([(19, 7), (20, 7), (20, 9), (19, 9)])
exit_bottom = simulation.add_exit_stage([(19, 1), (20, 1), (20, 3), (19, 3)])

journey_top    = simulation.add_journey(jps.JourneyDescription([exit_top]))
journey_bottom = simulation.add_journey(jps.JourneyDescription([exit_bottom]))

positions = jps.distributions.distribute_by_number(
    polygon=shapely.Polygon([(0.5, 0.5), (4, 0.5), (4, 9.5), (0.5, 9.5)]),
    number_of_agents=20,
    distance_to_agents=0.4,
    distance_to_polygon=0.2,
    seed=1,
)
agent_ids = [
    simulation.add_agent(
        jps.CollisionFreeSpeedModelAgentParameters(
            journey_id=journey_top,
            stage_id=exit_top,
            position=pos,
            radius=0.12,
        )
    )
    for pos in positions
]

rerouted = False
while simulation.agent_count() > 0 and simulation.iteration_count() < 10_000:
    if not rerouted and simulation.iteration_count() == 200:
        for agent_id in agent_ids[len(agent_ids) // 2 :]:
            simulation.switch_agent_journey(agent_id, journey_bottom, exit_bottom)
        rerouted = True
    simulation.iterate()

print(f"Done in {simulation.iteration_count()} iterations.")


In [ ]:
from jupedsim.internal.notebook_utils import animate, read_sqlite_file

trajectory_data, walkable_area = read_sqlite_file(trajectory_file)
animate(trajectory_data, walkable_area)


## Try one change

Change the re-route threshold from iteration 200 to iteration 50 (agents
re-route much earlier, so fewer reach the top exit before the switch):

```python
if not rerouted and simulation.iteration_count() == 50:
```

Or re-route only the first quarter instead of the second half:

```python
for agent_id in agent_ids[: len(agent_ids) // 4]:
```


## See also

- {doc}`Journeys and Transitions </notebooks/fundamentals/07_journeys_transitions>` – building journeys and transitions
- {doc}`Exit Stage </notebooks/fundamentals/03_exits>` – exit stage reference
- [JuPedSim scenario gallery](https://scenarios.jupedsim.org)
